In [95]:

import os  #--->  Talking to our operating system. Allows you to be able to read and write different types of files in your system.
import json #---> API response in going to be in JSON format, the JSON library helps us handle all that. 
import requests # ---> Allows us send request to the API. 
import pandas as pd  # ---> The pandas library helps in pandas and performing data transformations in tables(DataFrames, Python)
import pyodbc as connector # ---> Allows to connect to mssql programmatically. All the manual that could be done in mssql server is done here in with python. Load data from python via the pyodbc library
from dotenv import load_dotenv  #---> Allows us to load secrets from a dotenv file, safely. 

In [96]:
load_dotenv() # Gives the permission to load sensitive details in the .env file into this
              # this environment

API_KEY = os.getenv("API_KEY") # Get the API Key
API_HOST = os.getenv("API_HOST") # Get the APi Host 
SEASON = os.getenv("SEASON") # Specifies the season
LEAGUE_ID = os.getenv("LEAGUE_ID")
#print(API_KEY, "", API_HOST)


In [97]:
url = "https://v3.football.api-sports.io/standings"


headers = {
	"x-rapidapi-key": API_KEY,
	"x-rapidapi-host": API_HOST
}
querystring = {
     "league": LEAGUE_ID,
     "season": SEASON 
}



-------------------- EXTRACT ----------------------------

In [98]:
# This is sending a request to the api, using the request.get() function 
# and supplying the necessary parameters.

response = requests.get(url=url,
                        headers=headers,
                        params=querystring
                        )

In [99]:
# load the response to another variable 
payload = response.json()

In [100]:
payload

{'get': 'standings',
 'parameters': {'league': '39', 'season': '2024'},
 'errors': [],
 'results': 1,
 'paging': {'current': 1, 'total': 1},
 'response': [{'league': {'id': 39,
    'name': 'Premier League',
    'country': 'England',
    'logo': 'https://media.api-sports.io/football/leagues/39.png',
    'flag': 'https://media.api-sports.io/flags/gb-eng.svg',
    'season': 2024,
    'standings': [[{'rank': 1,
       'team': {'id': 40,
        'name': 'Liverpool',
        'logo': 'https://media.api-sports.io/football/teams/40.png'},
       'points': 84,
       'goalsDiff': 45,
       'group': 'Premier League',
       'form': 'DLDLW',
       'status': 'same',
       'description': 'Champions League',
       'all': {'played': 38,
        'win': 25,
        'draw': 9,
        'lose': 4,
        'goals': {'for': 86, 'against': 41}},
       'home': {'played': 19,
        'win': 14,
        'draw': 4,
        'lose': 1,
        'goals': {'for': 42, 'against': 16}},
       'away': {'played': 19,

-------------------------TRANSFORM ----------------------
Most of the field needed for this are hidden deep inside the json. Take this as navigating through different folders and directories on your computer. We dont want everything. Just specific things. 

Note: JSON are made up of keys and values. Keys on the left. The values are on the right.

The response key holds a list of nested JSON Data that's needed for this project

In [101]:
standing_list = payload['response'][0]['league']['standings'][0]

----------------------------- 2. TRANSFORM .............................
Parse API response into dataframe

In [102]:
rows = []
column_names = ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goal_diff', 'points', 'form' ]

#Loop through the list and extract the needed fields.
for club in standing_list:
    season          = 2024
    position        = club['rank']
    team_id         = club['team']['id']
    team            = club['team']['name']
    played          = club['all']['played']
    won             = club['all']['win']
    draw            = club['all']['draw']
    lost            = club['all']['lose']
    goals_for       = club['all']['goals']['for']
    goals_against   = club['all']['goals']['against']
    goal_diff       = club['goalsDiff']
    points          = club['points']
    form            = club['form']

    # This is a tuple: A row of data. We are packing everything into this tuple. And with a tuple, you cannot make any changes to it this data once it becomes tuple
    tuple_of_club_record = (season, position, team_id, team, played, won, draw, lost, goals_for,goals_against, goal_diff, points, form)

    # Append this tuple to the empty rows list variable
    rows.append(tuple_of_club_record)

In [104]:
# Put both the columns and rows in a pandas DataFrame
df = pd.DataFrame(rows, columns=column_names)
df

,season,position,team_id,team,played,won,draw,lost,goals_for,goals_against,goal_diff,points,form
0,2024,1,40,Liverpool,38,25,9,4,86,41,45,84,DLDLW
1,2024,2,42,Arsenal,38,20,14,4,69,34,35,74,WWDLD
2,2024,3,50,Manchester City,38,21,8,9,72,44,28,71,WWDWW
3,2024,4,49,Chelsea,38,20,9,9,64,43,21,69,WWLWW
4,2024,5,34,Newcastle,38,20,6,12,68,47,21,66,LLWDW
5,2024,6,66,Aston Villa,38,19,9,10,58,51,7,66,LWWWL
6,2024,7,65,Nottingham Forest,38,19,8,11,58,46,12,65,LWDDL
7,2024,8,51,Brighton,38,16,13,9,66,59,7,61,WWWDW
8,2024,9,35,Bournemouth,38,15,11,12,58,46,12,56,WLLWD
9,2024,10,55,Brentford,38,16,8,14,66,57,9,56,DLWWW


In [115]:
# Data Quality check 
if len(rows) !=20:
    raise ValueError("Data quality check failed: Expected 20 teams")

------------------------- LOAD DATA into MSSQL Server -------------------------

In [106]:
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'SQL Server Native Client RDA 11.0', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [107]:
# Establish connection to SQL Server using Windows authentication
import pyodbc

db_connection = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=gideon-oquongud\SQLEXPRESS;"
    "DATABASE=premier_league_standing;"
    "Trusted_Connection=yes;"
)


<>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\PUCHEO\AppData\Local\Temp\ipykernel_22700\4038232356.py:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  "SERVER=gideon-oquongud\SQLEXPRESS;"


In [ ]:
 # Initialize cursor for executing SQL commands
cursor = db_connection.cursor()

[Success] - Connection to database is succesful!!


In [109]:
# Validate target table existence before performing any load operation

sql_table = 'standings'

cursor.execute("""
        SELECT 1 
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_NAME = ?
               """, (sql_table,))

if cursor.fetchone() is None:
    raise SystemExit(f"This table '{sql_table}' is NOT found...please create it...")
else: 
    print(f"[SUCCESS] - This table '{sql_table}' exist! Continue to the next phase!")

[SUCCESS] - This table 'standings' exist! Continue to the next phase!


In [110]:
# Align DataFrame schema with target SQL table structure to prevent column mismatch issues
table_cols = ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goal_diff', 'points', 'form' ]

standings_df = df[table_cols] # Subset DataFrame to only required columns in correct order

In [111]:
# Convert DataFrame rows into tuples for efficient bulk database operations
standings_records_tuples = standings_df.itertuples(index=False, name=None)

# Materialize iterator into list for batch execution
list_of_standings_records_tuples = list(standings_records_tuples)

In [117]:
# Define UPSERT (MERGE) logic: update existing records or insert new ones based on business key
merge_SQL = f"""
    MERGE INTO {sql_table} AS target
    USING (VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)) AS source
    (season, position, team_id, team, played, won, draw, lost, goals_for, goals_against, goal_diff, points, form)

    ON target.team_id = source.team_id AND target.season = source.season

    WHEN MATCHED THEN 
        UPDATE SET 
            position        = source.position,
            team            = source.team,
            played          = source.played, 
            won             = source.won,
            draw            = source.draw, 
            lost            = source.lost, 
            goals_for       = source.goals_for, 
            goals_against   = source.goals_against, 
            goal_diff       = source.goal_diff, 
            points          = source.points,
            form            = source.form
    WHEN NOT MATCHED THEN 
    INSERT (season, position, team_id, team, played, won, draw, lost,goals_for, goals_against, goal_diff, points, form)
    VALUES(source.season, source.position, source.team_id, source.team, source.played, source.won, source.draw, source.lost, source.goals_for, source.goals_against, source.goal_diff, source.points, source.form);
    """


In [ ]:
no_of_rows_uploaded_mssql = len(list_of_standings_records_tuples)
no_of_rows_uploaded_mssql

20

In [114]:
# Execute batch UPSERT with transaction handling, rollback on failure, and guaranteed resource cleanup

try:
    cursor = db_connection.cursor()
    
    cursor.executemany(merge_SQL, list_of_standings_records_tuples)
    db_connection.commit()
    print(f"[SUCCESS] - Upsert attempted for {no_of_rows_uploaded_mssql} ")
except Exception as e:
    print(f"[ERROR] - Rolled back due to this....{e}")

    try:
        db_connection.rollback()
    except Exception:
        pass

finally:
    try:
        cursor.close()
    except Exception:
        pass
    try:
        db_connection.close()
    except Exception:
        pass

    print("All database connections now closed. \n\n Clean up completed.")

[SUCCESS] - Upsert attempted for 20 
All database connections now closed. 

 Clean up completed.
